In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
! pip install dagshub mlflow

# Dagshub/Mlflow initialization

In [3]:
import dagshub
import mlflow
import mlflow.sklearn
import os

TOKEN = '773e6d0f45f4f8c8ed729b693f548555a4de31f8'

os.environ['DAGSHUB_USER_TOKEN'] = TOKEN
dagshub.auth.add_app_token(TOKEN)

dagshub.init(repo_owner='sbolk23', repo_name='IEEE-CIS-Fraud-Detection-Kaggle-Competition', mlflow=True)

The added token already exists in the token cache, skipping


Accessing as sbolk23

Initialized MLflow to track repo "sbolk23/IEEE-CIS-Fraud-Detection-Kaggle-Competition"

Repository sbolk23/IEEE-CIS-Fraud-Detection-Kaggle-Competition initialized!

In [4]:
TARGET = 'isFraud'

In [5]:
df_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
df_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

In [6]:
print('train_trancation shape:', df_transaction.shape)
print('train_identity shape:',   df_identity.shape)

train_trancation shape: (590540, 394)
train_identity shape: (144233, 41)


In [7]:
df = pd.merge(df_transaction, df_identity, on='TransactionID', how='left')

In [8]:
print('Data count:',  df[TARGET].shape[0])
print('Event count:', df[TARGET].sum())

Data count: 590540
Event count: 20663


In [9]:
sorted_df = df.sort_values(by='TransactionDT')

train_size = int(sorted_df.shape[0] * .7)
val_size   = int(sorted_df.shape[0] * .15)

train_df = sorted_df.iloc[:train_size]
val_df   = sorted_df.iloc[train_size: train_size + val_size]
test_df  = sorted_df.iloc[train_size + val_size:]

print('Train shape:',      train_df.shape, '\nTrain prevalence:',      train_df[TARGET].sum() / train_df.shape[0], '\n')
print('Validation shape:', val_df.shape,   '\nValidation prevalence:', val_df[TARGET].sum()   / val_df.shape[0],   '\n')
print('Test shape:',       test_df.shape,  '\nTrain prevalence:',      test_df[TARGET].sum()  / test_df.shape[0],  '\n')

Train shape: (413378, 434) 
Train prevalence: 0.03516878014795175 

Validation shape: (88581, 434) 
Validation prevalence: 0.03434145019812375 

Test shape: (88581, 434) 
Train prevalence: 0.03480430340592226 



In [19]:
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_val   = val_df.drop(columns=[TARGET])
y_val   = val_df[TARGET]

X_test  = test_df.drop(columns=[TARGET])
y_test  = test_df[TARGET]

In [11]:
# X_val   = val_df.drop(columns=[TARGET])
# y_val   = val_df[TARGET]

# X_test  = test_df.drop(columns=[TARGET])
# y_test  = test_df[TARGET]

In [12]:
# # Fix outliers
# outliers = train_df[train_df['TransactionAmt'] >= 30000]

# # Remove outliers
# train_df = train_df[train_df['TransactionAmt'] < 30000]

# # Split train_df into X_train and y_train
# X_train = train_df.drop(columns=[TARGET])
# y_train = train_df[TARGET]

# Data Cleaning

# Uninformative Columns

In [36]:
# TransactionID and TransactionDT are not informative feature so we remove it

irrelevant_cols = [
    'TransactionID',
    'TransactionDT',
]

# Preprocessing Pipeline

In [37]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Drop irrelevant columns
irrelevant_col_dropper = ColumnTransformer(
    transformers=[
        ('drop', 'drop', irrelevant_cols),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
).set_output(transform='pandas')

# Impute & Encode numeric and categorical columns
num_pipeline = Pipeline([
     ('imputer', SimpleImputer(strategy='constant', fill_value=-999)),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])

prep = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, make_column_selector(dtype_include='number')),
        ('cat', cat_pipeline, make_column_selector(dtype_include='object')),
    ],
    verbose_feature_names_out=False,
).set_output(transform='pandas')

# Full preprocessor pipeline
preprocessor = Pipeline([
    ('dropper',         irrelevant_col_dropper),
    ('prep',            prep),
])

# Full Pipeline

In [38]:
from sklearn.tree import DecisionTreeClassifier
# from cuml.tree import DecisionTreeClassifier

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model',        DecisionTreeClassifier()),
])

preprocessor_configs = {
    'prep__num__imputer__strategy':   ['constant'],
    'prep__num__imputer__fill_value': [-999],

    'prep__cat__imputer__strategy':   ['constant'],
    'prep__cat__imputer__fill_value': ['missing'],
    
    'prep__cat__encoder__handle_unknown': ['use_encoded_value'],
    'prep__cat__encoder__unknown_value':  [-1],
}

model_configs = {
    'max_depth':         [5, 10, 12, 13, 14, 15, None],
    'min_samples_leaf':  [400, 500, 550, 600],
    'class_weight':      ['balanced'],
    'criterion':         ['gini', 'entropy'],
    'max_features':      [None],
    'random_state':      [1337],
}

# MLflow Logging 

# Model Parameters & Metrics Logging Helper function

In [39]:
from sklearn.metrics import (
    roc_auc_score, 
    average_precision_score, 
    log_loss, 
    brier_score_loss, 
    f1_score, 
    precision_score, 
    recall_score, 
    balanced_accuracy_score, 
    matthews_corrcoef,
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    roc_curve
)

def log_model_parameters_metrics(y_true, y_prob, prefix):
    
    # Best threshold for current predicition probabilities
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    best_threshold = thresholds[min(np.argmax(f1_scores), len(thresholds) - 1)]

    y_pred = (y_prob >= best_threshold).astype(int)

    
    print(f'Logging model metrics on {prefix} set...')
    
    # Metrics independent from threshold
    auc_roc         = roc_auc_score(y_true, y_prob)
    auc_pr          = average_precision_score(y_true, y_prob)
    model_log_loss  = log_loss(y_true, y_prob)
    model_brier     = brier_score_loss(y_true, y_prob)

    mlflow.log_metric(f'{prefix}_auc_roc',  auc_roc)
    mlflow.log_metric(f'{prefix}_auc_pr',   auc_pr)
    mlflow.log_metric(f'{prefix}_log_loss', model_log_loss)
    mlflow.log_metric(f'{prefix}_brier',    model_brier)

    
    # Metrics dependent on best threshold
    model_f1        = f1_score(y_true, y_pred)
    model_precision = precision_score(y_true, y_pred)
    model_recall    = recall_score(y_true, y_pred)
    
    mlflow.log_metric(f'{prefix}_f1',        model_f1)
    mlflow.log_metric(f'{prefix}_precision', model_precision)
    mlflow.log_metric(f'{prefix}_recall',    model_recall)

    # Best threshold
    mlflow.log_metric(f'{prefix}_best_threshold', best_threshold)

    
    # Log confusion matrix and classification report
    cm     = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred)
    
    mlflow.log_text(f'Confusion Matrix:\n{cm}',          f'{prefix}_confusion_matrix.txt')
    mlflow.log_text(f'Classification Report:\n{report}', f'{prefix}_classification_report.txt')


# Model Metrics Curves Logging Helper Function

In [40]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, auc

def log_model_metrics_curves(y_true, y_prob, prefix):
    fpr, tpr, roc_thresholds = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)

    # Log ROC curve
    fig_roc, ax_roc = plt.subplots(figsize=(6, 5))
    ax_roc.plot(fpr, tpr, label=f'ROC AUC = {roc_auc:.4f}')
    ax_roc.plot([0, 1], [0, 1], linestyle='--')
    ax_roc.set_xlabel('False Positive Rate')
    ax_roc.set_ylabel('True Positive Rate')
    ax_roc.set_title(f'ROC Curve - {prefix}')
    ax_roc.legend(loc='lower right')
    mlflow.log_figure(fig_roc, f'{prefix}_roc_curve.png')
    plt.close(fig_roc)

    # Log Precision-Recall curve
    precisions, recalls, pr_thresholds = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(recalls, precisions)

    fig_pr, ax_pr = plt.subplots(figsize=(6, 5))
    ax_pr.plot(recalls, precisions, label=f'PR AUC = {pr_auc:.4f}')
    ax_pr.set_xlabel('Recall')
    ax_pr.set_ylabel('Precision')
    ax_pr.set_title(f'Precision-Recall Curve - {prefix}')
    ax_pr.legend(loc='lower left')
    mlflow.log_figure(fig_pr, f'{prefix}_pr_curve.png')
    plt.close(fig_pr)


    # Best threshold for current predicition probabilities
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    best_threshold = thresholds[min(np.argmax(f1_scores), len(thresholds) - 1)]

    y_pred = (y_prob >= best_threshold).astype(int)
    cm     = confusion_matrix(y_true, y_pred)
    
    # Log Confusion Matrix
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"Confusion Matrix - {prefix}")
    mlflow.log_figure(fig, f"{prefix}_confusion_matrix.png")

In [46]:
from sklearn.model_selection import ParameterGrid, GridSearchCV
from sklearn.base import clone

model_configs = {
    'max_depth':        [None,],
    'min_samples_leaf': [700],
    'class_weight':     ['balanced'],
    'criterion':        ['gini',],
    'random_state':     [1337],
}

# 'ccp_alpha':        [0, 5e-6, 6e-6, 7e-6, 8e-6, 9e-6, 1e-5, 3e-5, 5e-5, 6e-5],
# 'max_leaf_nodes':   [50, 100, 150, 200,],
# 'ccp_alpha': [0.0, 1e-5, 3e-5, 5e-5, 1e-4],
# 'max_features':     [.1, .3, .5, .7, .9, 1],
# 'min_samples_split':[1, 10, 50, 100],

for prep_param in ParameterGrid(preprocessor_configs):
    preprocessor = clone(full_pipeline.named_steps['preprocessor'])
    preprocessor.set_params(**prep_param)

    print('Starting fitting & transforming X_train, X_val...')

    X_train_t = preprocessor.fit_transform(X_train, y_train)
    print('fit_transform X_train finished...')

    X_val_t   = preprocessor.transform(X_val)
    print('transform X_val finished...')

    print(prep_param)

    for model_param in ParameterGrid(model_configs):    
        model = clone(full_pipeline.named_steps['model'])
        model.set_params(**model_param)

        print('Currently running model with params:', model_param)
        
        print('Starting training model on X_train_t...')
        model.fit(X_train_t, y_train)
        print('Finished training model on X_train_t...')

        print('Starting predicting on X_train_t...')
        y_train_prob = model.predict_proba(X_train_t)[:, 1]
        print('Finished predicting on X_train_t...')
        
        print('Starting predicting on X_val_t...')
        y_val_prob   = model.predict_proba(X_val_t)[:, 1]
        print('Finished predicting on X_val_t...')

        print('train_roc_score:', roc_auc_score(y_train, y_train_prob))
        print('test_roc_score:',  roc_auc_score(y_val, y_val_prob))


Starting fitting & transforming X_train, X_val...
fit_transform X_train finished...
transform X_val finished...
{'prep__cat__encoder__handle_unknown': 'use_encoded_value', 'prep__cat__encoder__unknown_value': -1, 'prep__cat__imputer__fill_value': 'missing', 'prep__cat__imputer__strategy': 'constant', 'prep__num__imputer__fill_value': -999, 'prep__num__imputer__strategy': 'constant'}
Currently running model with params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 700, 'random_state': 1337}
Starting training model on X_train_t...
Finished training model on X_train_t...
Starting predicting on X_train_t...
Finished predicting on X_train_t...
Starting predicting on X_val_t...
Finished predicting on X_val_t...
train_roc_score: 0.9066450592086427
test_roc_score: 0.8587093764759013


In [ ]:
# from sklearn.model_selection import ParameterGrid, GridSearchCV
# from sklearn.base import clone

# print('Start MLflow logging')


# for prep_param in ParameterGrid(preprocessor_configs):
#     experiment_name = (
#         f'IEEE-CIS_Fraud_Detection_Decision_Tree_Training'
#         f'__prep_v0'
#     )
    
#     mlflow.set_experiment(experiment_name)
    
#     preprocessor = clone(full_pipeline.named_steps['preprocessor'])
#     preprocessor.set_params(**prep_param)

#     print('Starting fitting & transforming X_train, X_val...')

#     X_train_t = preprocessor.fit_transform(X_train, y_train)
#     print('fit_transform X_train finished...')

#     X_val_t   = preprocessor.transform(X_val)
#     print('transform X_val finished...')

#     print(prep_param)
#     with mlflow.start_run(run_name='DT_Cleaning'):
#         # Logging Data Split 
#         mlflow.log_param('split_method',                'Chronological')
        
#         mlflow.log_param('train_df_ratio',              len(train_df) / len(df))
#         mlflow.log_param('val_df_ratio',                len(val_df)   / len(df))
#         mlflow.log_param('test_df_ratio',               len(test_df)  / len(df))

#         mlflow.log_metric('train_df_prevalence',        train_df[TARGET].sum() / train_df.shape[0])
#         mlflow.log_metric('val_df_prevalence',          val_df[TARGET].sum()   / val_df.shape[0])
#         mlflow.log_metric('test_df_prevalence',         test_df[TARGET].sum()  / test_df.shape[0])
        
        
#         # Logging cleaning parameters
#         mlflow.log_param('irrelevant_cols',             str(irrelevant_cols))
#         mlflow.log_param('n_irrelevant_cols_dropped',   len(irrelevant_cols))
#         mlflow.log_param('num_imputer_strategy',        prep_param['prep__num__imputer__strategy'])
#         mlflow.log_param('num_imputer_fill_value',      prep_param['prep__num__imputer__fill_value'])
        
#         mlflow.log_param('cat_imputer_strategy',        prep_param['prep__cat__imputer__strategy'])
#         mlflow.log_param('cat_imputer_fill_value',      prep_param['prep__cat__imputer__fill_value'])
        
#         mlflow.log_param('cat_encoder',                 'OrdinalEncoder')

#         # Logging cleaning metrics
#         mlflow.log_metric('X_train_rows_before', X_train.shape[0])
#         mlflow.log_metric('X_train_cols_before', X_train.shape[1])
#         mlflow.log_metric('X_train_cols_after',  X_train_t.shape[1])
#         mlflow.log_metric('X_val_rows_before',   X_val.shape[0])
#         mlflow.log_metric('X_val_cols_before',   X_val.shape[1])
#         mlflow.log_metric('X_val_cols_after',    X_val_t.shape[1])
#         mlflow.log_metric('fraud_rate_train',    y_train.mean())
#         mlflow.log_metric('fraud_rate_val',      y_val.mean())


#     for model_param in ParameterGrid(model_configs):    
#         model = clone(full_pipeline.named_steps['model'])
#         model.set_params(**model_param)

#         print('Currently running model with params:', model_param)
        
#         print('Starting training model on X_train_t...')
#         model.fit(X_train_t, y_train)
#         print('Finished training model on X_train_t...')

#         print('Starting predicting on X_train_t...')
#         y_train_prob = model.predict_proba(X_train_t)[:, 1]
#         print('Finished predicting on X_train_t...')
        
#         print('Starting predicting on X_val_t...')
#         y_val_prob   = model.predict_proba(X_val_t)[:, 1]
#         print('Finished predicting on X_val_t...')

#         print('train_auc_roc_score:', roc_auc_score(y_train, y_train_prob))
#         print('test_auc_roc_score:',  roc_auc_score(y_val, y_val_prob))

#         # Prepare for logging
#         max_depth        = model_param['max_depth']
#         min_samples_leaf = model_param['min_samples_leaf']
#         criterion        = model_param['criterion']
#         max_features     = model_param['max_features']
#         class_weight     = model_param['class_weight']

#         run_name = (
#             f'DT_Training'
#             f'__depth_{max_depth}'
#             f'__min_samples_leaf_{min_samples_leaf}'
#             f'__criterion_{criterion}'
#             f'__max_features_{max_features}'
#             f'__class_weight_{class_weight}'
#         )

#         # Start logging
#         with mlflow.start_run(run_name=run_name):
#             print('Logging model training results on mlflow...')

#             print('Loggin model parameters...')
#             for key, val in model_param.items():
#                 mlflow.log_param(key, val)
            
#             log_model_parameters_metrics(y_train, y_train_prob, 'train')
#             log_model_parameters_metrics(y_val,   y_val_prob,   'val')
            
#             log_model_metrics_curves(y_train, y_train_prob, 'train')
#             log_model_metrics_curves(y_val,   y_val_prob,   'val')

#             # Log full pipeline
#             full_pipe = Pipeline([
#                 ('preprocessor', preprocessor),
#                 ('model',        model)
#             ])
            
#             model_info = mlflow.sklearn.log_model(
#                 sk_model=full_pipe,
#                 artifact_path='pipeline'
#             )
            
#             mlflow.set_tag('model_id', model_info.model_id)
#             mlflow.set_tag('model_type', 'DecisionTreeClassifier')

#             # Log feature importance
#             importance_df = pd.DataFrame({
#                 'feature':    preprocessor.get_feature_names_out(),
#                 'importance': model.feature_importances_,
#             }).sort_values('importance', ascending=False)
            
#             importance_df.to_csv('feature_importances.csv', index=False)
#             mlflow.log_artifact('feature_importances.csv')
            
#             fig, ax = plt.subplots(figsize=(10, 8))
#             top20 = importance_df.head(20)
#             ax.barh(top20['feature'], top20['importance'])
#             ax.set_title('Top 20 Feature Importances')
#             ax.set_xlabel('Importance')
#             plt.tight_layout()
#             mlflow.log_figure(fig, 'top20_feature_importances.png')
#             plt.close()